# TF-IDF + Lineer SVM - Turkce Metin Siniflandirma Kiyaslamasi

**Katman:** Klasik taban cizgisi

Kucuk veride hala guclu klasik taban.

Gorev: Turkce deprem ihbar metni -> 5 ust kategori (+ opsiyonel 1-5 aciliyet).  
Veri: `data/` (bkz `data/README.md`).  
Butun defterler `common.py` uzerinden AYNI bolme ve AYNI metrikleri kullanir.  
Metrik: Dogruluk, makro-F1, sinif bazli P/R, konfuzyon matrisi, **yanlis negatif orani**.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/04_turkce_metin
    %pip install -q scikit-learn
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
import common as C

train_df, val_df, test_df = C.load_text_dataset()
labels, l2i, i2l = C.label_maps(train_df)
print('siniflar:', labels)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
y_test = test_df['label'].map(l2i).values

## 2. Ozellik + model

In [ ]:
MODEL_NAME = "tfidf_svm"
USE_ZEYREK = False   # True yaparsan BERTurk hattiyla ayni lemmatizasyon

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
clf = CalibratedClassifierCV(LinearSVC(class_weight='balanced', C=1.0))

Xtr_txt = train_df['text'].tolist(); Xte_txt = test_df['text'].tolist()
if USE_ZEYREK:
    Xtr_txt = C.zeyrek_lemmatize(Xtr_txt); Xte_txt = C.zeyrek_lemmatize(Xte_txt)

vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000, sublinear_tf=True)
Xtr = vec.fit_transform(Xtr_txt); Xte = vec.transform(Xte_txt)
ytr = train_df['label'].map(l2i).values
PARAMS = {'vectorizer': str(vec), 'clf': str(clf)}

## 3. Egit ve tahmin et

In [ ]:
clf.fit(Xtr, ytr)
y_pred = clf.predict(Xte)

## 4. Sonuc

In [ ]:
m = C.classification_metrics(y_test, y_pred, labels)
print('accuracy :', round(m['accuracy'], 4))
print('macro-F1 :', round(m['macro_f1'], 4))
print('macro yanlis-negatif orani:', round(m['macro_fnr'], 4))
print()
print(m['report'])
lb = C.save_result(MODEL_NAME, m, extra={'params': PARAMS})
lb

In [ ]:
import matplotlib.pyplot as plt
cm = np.array(m['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, int(v), ha='center', va='center')
ax.set_xlabel('tahmin'); ax.set_ylabel('gercek'); ax.set_title(f'{MODEL_NAME} konfuzyon')
plt.tight_layout(); plt.savefig(f'results/{MODEL_NAME}_cm.png', dpi=140); plt.show()